# Backtest Statistics

## Overview

This notebook demonstrates the namespace classes in `backtest_statistics` using a compact synthetic backtest.
- Problem: backtests from historical, cross-validated, or synthetic simulations need comparable statistics that reveal performance, asymmetric risk, capacity limits, execution sensitivity, and model overlay quality.
- Approach: compute grouped statistics from positions, returns, PnL, AUM, execution costs, benchmark returns, and classifier predictions so a strategy can be judged across complementary dimensions.
- General Characteristics: It summarizes sample coverage, capital usage, capacity, leverage, bet frequency, turnover, holding period, and market exposure.
- Performance: It reports raw dollar and return outcomes, including long-side contribution, annualized return, hit ratio, and hit/miss averages.
- Runs: It measures return concentration, drawdowns, and time under water to expose path-dependent downside risk.
- Implementation Shortfall: It compares performance with turnover, fees, slippage, and execution costs.
- Efficiency: It computes Sharpe, information, probabilistic Sharpe, and deflated Sharpe statistics.
- Classification Scores: It evaluates the meta-labeling overlay with accuracy, precision, recall, F1, and negative log-loss.

In [1]:
import numpy as np
import pandas as pd

In [2]:
from src.model_backtesting.backtest_statistics import (
    ClassificationScores,
    Efficiency,
    GeneralCharacteristics,
    ImplementationShortfall,
    Performance,
    Runs,
)

## Example Backtest Data

This cell defines the compact synthetic inputs used by the examples below.
- `position` is the target position series, where positive values are long, negative values are short, and zero is flat.
- `underlying_return` is the periodic return series of the underlying investment universe, generated from position direction and independent noise.
- `benchmark_return` is the periodic return series of the benchmark portfolio, generated from position direction and independent noise.
- `strategy_return` is the strategy's periodic return series, generated from active position state, target exposure, and independent strategy noise.
- `aum` is the assets-under-management series, expressed in dollars.
- `pnl` is the strategy's periodic dollar profit and loss.
- `position_value` is the target position converted into dollar exposure as a fraction of AUM.
- `traded_value` is the dollar value traded at each timestamp.
- `portfolio_value` is the compounded value path implied by the strategy returns.
- `broker_fees` is the explicit broker and exchange fee series.
- `slippage` is the implicit execution-cost series from trading away from the reference price.
- `execution_costs` is the total execution-cost series, combining broker fees and slippage.

In [3]:
dates = pd.bdate_range("2024-01-02", periods=60)
rng = np.random.default_rng(42)
target_exposure = 0.25

positions = pd.Series(
    np.repeat([0, 1, 0, -1, 0, 1, 0, -1, 0, 1, 0, -1], 5),
    index=dates,
    name="position",
)
underlying_returns = pd.Series(
    positions * 0.0008 + rng.normal(0.0001, 0.004, dates.shape[0]),
    index=dates,
    name="underlying_return",
)
benchmark_returns = pd.Series(
    positions * 0.0003 + rng.normal(0.0001, 0.003, dates.shape[0]),
    index=dates,
    name="benchmark_return",
)
strategy_noise = pd.Series(rng.normal(0, 0.004, dates.shape[0]), index=dates)
strategy_returns = pd.Series(
    target_exposure * (positions.abs() * 0.0025 + positions * strategy_noise),
    index=dates,
    name="strategy_return",
)
aum = pd.Series(
    1_000_000 + np.linspace(0, 50_000, dates.shape[0]),
    index=dates,
    name="aum",
)
pnl = (strategy_returns * aum).rename("pnl")
position_values = (positions * aum * target_exposure).rename("position_value")
traded_value = position_values.diff().abs().fillna(0).rename("traded_value")
portfolio_value = (1.0 + strategy_returns).cumprod().rename("portfolio_value")
broker_fees = (traded_value * 0.0001).rename("broker_fees")
slippage = (traded_value * 0.0003).rename("slippage")
execution_costs = (broker_fees + slippage).rename("execution_costs")

example_data = pd.concat(
    [
        positions,
        underlying_returns,
        benchmark_returns,
        strategy_returns,
        aum,
        pnl,
        position_values,
        traded_value,
        portfolio_value,
        broker_fees,
        slippage,
        execution_costs,
    ],
    axis=1,
)

example_data.head(12)

,position,underlying_return,benchmark_return,strategy_return,aum,pnl,position_value,traded_value,portfolio_value,broker_fees,slippage,execution_costs
2024-01-02,0,0.001319,-0.004949,0.000000,1.000000e+06,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
2024-01-03,0,-0.004060,-0.000905,0.000000,1.000847e+06,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
2024-01-04,0,0.003102,0.000588,0.000000,1.001695e+06,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
2024-01-05,0,0.003862,0.001859,0.000000,1.002542e+06,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
2024-01-08,0,-0.007704,0.002234,0.000000,1.003390e+06,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
2024-01-09,1,-0.004309,0.002780,0.000982,1.004237e+06,986.031530,251059.322034,251059.322034,1.000982,25.105932,75.317797,100.423729
2024-01-10,1,0.001411,-0.000646,0.002088,1.005085e+06,2098.921380,251271.186441,211.864407,1.003072,0.021186,0.063559,0.084746
2024-01-11,1,-0.000365,-0.000987,-0.000564,1.005932e+06,-567.107411,251483.050847,211.864407,1.002507,0.021186,0.063559,0.084746
2024-01-12,1,0.000833,0.002974,-0.000015,1.006780e+06,-14.851543,251694.915254,211.864407,1.002492,0.021186,0.063559,0.084746
2024-01-15,1,-0.002512,-0.000174,-0.000302,1.007627e+06,-303.876097,251906.779661,211.864407,1.002190,0.021186,0.063559,0.084746


## General Characteristics

This cell reports statistics that describe the shape and operating footprint of the backtest.
- `time_range` describes the start and end timestamps of the backtest sample.
- `average_aum` describes the average capital base used by the strategy.
- `capacity` describes the highest AUM that still meets the target risk-adjusted performance.
- `leverage` describes average gross dollar exposure relative to average AUM.
- `maximum_dollar_position_size` describes the largest gross dollar position taken by the strategy.
- `ratio_of_longs` describes the fraction of active positions that are long.
- `frequency_of_bets` describes how many independent bets are placed per year.
- `average_holding_period` describes the position-weighted average holding period in days.
- `annualized_turnover` describes annual traded dollar value relative to average AUM.
- `correlation_to_underlying` describes directional exposure to the underlying return series.

In [4]:
general_statistics = pd.Series(
    {
        "time_range": GeneralCharacteristics.time_range(strategy_returns),
        "average_aum": GeneralCharacteristics.average_aum(aum),
        "capacity": GeneralCharacteristics.capacity(
            aum=aum,
            risk_adjusted_performance=pd.Series(
                np.linspace(0.4, 1.0, dates.shape[0]),
                index=dates,
            ),
            target_performance=0.8,
        ),
        "leverage": GeneralCharacteristics.leverage(position_values, aum),
        "maximum_dollar_position_size": (
            GeneralCharacteristics.maximum_dollar_position_size(position_values)
        ),
        "ratio_of_longs": GeneralCharacteristics.ratio_of_longs(positions),
        "frequency_of_bets": GeneralCharacteristics.frequency_of_bets(positions),
        "average_holding_period": (
            GeneralCharacteristics.average_holding_period(positions)
        ),
        "annualized_turnover": GeneralCharacteristics.annualized_turnover(
            traded_value,
            aum,
        ),
        "correlation_to_underlying": (
            GeneralCharacteristics.correlation_to_underlying(
                strategy_returns,
                underlying_returns,
            )
        ),
    }
)

general_statistics

time_range                      (2024-01-02 00:00:00, 2024-03-25 00:00:00)
average_aum                                                      1025000.0
capacity                                                         1050000.0
leverage                                                          0.125258
maximum_dollar_position_size                                      262500.0
ratio_of_longs                                                         0.5
frequency_of_bets                                                30.804217
average_holding_period                                                 7.0
annualized_turnover                                              12.123942
correlation_to_underlying                                         0.021222
dtype: object

## Performance

This cell reports unadjusted dollar and return performance statistics.
- `pnl` measures total dollar profit and loss.
- `pnl_from_long_positions` measures dollar profit and loss generated while the strategy is long.
- `annualized_rate_of_return` converts the compounded return path into an annualized return.
- `hit_ratio` measures the fraction of active return observations with positive returns.
- `average_return_from_hits` measures the average return among winning active return observations.
- `average_return_from_misses` measures the average return among losing active return observations.

In [5]:
active_returns = strategy_returns[positions != 0]

performance_statistics = pd.Series(
    {
        "pnl": Performance.pnl(pnl),
        "pnl_from_long_positions": Performance.pnl_from_long_positions(
            pnl,
            positions,
        ),
        "annualized_rate_of_return": Performance.annualized_rate_of_return(
            strategy_returns,
            periods_per_year=252,
        ),
        "hit_ratio": Performance.hit_ratio(active_returns),
        "average_return_from_hits": Performance.average_return_from_hits(active_returns),
        "average_return_from_misses": Performance.average_return_from_misses(
            active_returns,
        ),
    }
)

performance_statistics

pnl                           10063.662190
pnl_from_long_positions        8928.868767
annualized_rate_of_return         0.041791
hit_ratio                         0.666667
average_return_from_hits          0.000863
average_return_from_misses       -0.000749
dtype: float64

## Runs

This cell reports path-dependent risk and concentration statistics.
- `hhi_positive_returns` measures whether gains are concentrated in a few positive-return active observations.
- `hhi_negative_returns` measures whether losses are concentrated in a few negative-return active observations.
- `hhi_time_between_bets` measures whether trading activity is concentrated in a few time periods.
- `max_drawdown` measures the worst observed loss from a high-water mark.
- `max_time_under_water` measures the longest observed recovery time from a high-water mark.
- `95_percentile_drawdown` measures a severe but less extreme drawdown level.
- `95_percentile_time_under_water` measures a severe but less extreme recovery-time level.

In [6]:
active_returns = strategy_returns[positions != 0]

drawdown = Runs.drawdown(portfolio_value)
time_under_water = Runs.time_under_water(portfolio_value)

runs_statistics = pd.Series(
    {
        "hhi_positive_returns": Runs.hhi_positive_returns(active_returns),
        "hhi_negative_returns": Runs.hhi_negative_returns(active_returns),
        "hhi_time_between_bets": Runs.hhi_time_between_bets(active_returns),
        "max_drawdown": drawdown.max(),
        "max_time_under_water": time_under_water.max(),
        "95_percentile_drawdown": Runs.percentile_drawdown(portfolio_value),
        "95_percentile_time_under_water": (
            Runs.percentile_time_under_water(portfolio_value)
        ),
    }
)

runs_statistics

hhi_positive_returns              0.024272
hhi_negative_returns              0.087069
hhi_time_between_bets             0.000000
max_drawdown                      0.003870
max_time_under_water              0.090349
95_percentile_drawdown            0.003432
95_percentile_time_under_water    0.087337
dtype: float64

## Implementation Shortfall

This cell reports execution-cost sensitivity statistics.
- `broker_fees_per_turnover` measures explicit broker fees per unit of turnover.
- `average_slippage_per_turnover` measures implicit slippage costs per unit of turnover.
- `dollar_performance_per_turnover` measures dollar performance earned for each dollar traded.
- `return_on_execution_costs` measures how large performance is relative to total execution costs.

In [7]:
shortfall_statistics = pd.Series(
    {
        "broker_fees_per_turnover": (
            ImplementationShortfall.broker_fees_per_turnover(
                broker_fees,
                traded_value,
            )
        ),
        "average_slippage_per_turnover": (
            ImplementationShortfall.average_slippage_per_turnover(
                slippage,
                traded_value,
            )
        ),
        "dollar_performance_per_turnover": (
            ImplementationShortfall.dollar_performance_per_turnover(
                pnl,
                traded_value,
            )
        ),
        "return_on_execution_costs": (
            ImplementationShortfall.return_on_execution_costs(
                pnl,
                execution_costs,
            )
        ),
    }
)

shortfall_statistics

broker_fees_per_turnover           0.000100
average_slippage_per_turnover      0.000300
dollar_performance_per_turnover    0.003564
return_on_execution_costs          8.909237
dtype: float64

## Efficiency

This cell reports risk-adjusted and trial-adjusted performance statistics.
- `sharpe_ratio` measures average return per unit of return volatility in the observed sampling frequency.
- `annualized_sharpe_ratio` measures the Sharpe ratio after annualizing by the configured observation frequency.
- `information_ratio` measures excess return versus the benchmark per unit of tracking error.
- `probabilistic_sharpe_ratio` estimates whether the observed Sharpe ratio remains meaningful after accounting for sample length and non-normal returns.
- `deflated_sharpe_ratio` estimates whether the observed Sharpe ratio remains meaningful after also accounting for multiple position-rule trials.

In [8]:
trial_sharpe_ratios = pd.Series([0.2, 0.4, 0.1, 0.35, 0.25])

efficiency_statistics = pd.Series(
    {
        "sharpe_ratio": Efficiency.sharpe_ratio(strategy_returns),
        "annualized_sharpe_ratio": Efficiency.annualized_sharpe_ratio(
            strategy_returns,
            periods_per_year=252,
        ),
        "information_ratio": Efficiency.information_ratio(
            strategy_returns,
            benchmark_returns,
            periods_per_year=252,
        ),
        "probabilistic_sharpe_ratio": (
            Efficiency.probabilistic_sharpe_ratio(strategy_returns)
        ),
        "deflated_sharpe_ratio": Efficiency.deflated_sharpe_ratio(
            strategy_returns,
            trial_sharpe_ratios,
        ),
    }
)

efficiency_statistics

sharpe_ratio                  0.227648
annualized_sharpe_ratio       3.613802
information_ratio             4.168058
probabilistic_sharpe_ratio    0.955843
deflated_sharpe_ratio         0.738428
dtype: float64

## Classification Scores

This cell reports classification metrics for the meta-labeling overlay.
- `accuracy` measures the overall fraction of correct take/pass decisions.
- `precision` measures the fraction of predicted positive decisions that are correct.
- `recall` measures the fraction of actual positive decisions that are recovered.
- `f1_score` measures the harmonic mean of precision and recall.
- `negative_log_loss` evaluates predicted probabilities, not only the final class labels.

In [9]:
y_true = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 1, 0, 0, 1, 0, 0, 0, 0])
y_pred_proba = np.array(
    [
        [0.15, 0.85],
        [0.25, 0.75],
        [0.35, 0.65],
        [0.60, 0.40],
        [0.70, 0.30],
        [0.45, 0.55],
        [0.80, 0.20],
        [0.90, 0.10],
        [0.65, 0.35],
        [0.75, 0.25],
    ]
)

classification_statistics = pd.Series(
    {
        "accuracy": ClassificationScores.accuracy(y_true, y_pred),
        "precision": ClassificationScores.precision(y_true, y_pred),
        "recall": ClassificationScores.recall(y_true, y_pred),
        "f1_score": ClassificationScores.f1_score(y_true, y_pred),
        "negative_log_loss": ClassificationScores.negative_log_loss(
            y_true,
            y_pred_proba,
            labels=[0, 1],
        ),
    }
)

classification_statistics

accuracy             0.700000
precision            0.750000
recall               0.600000
f1_score             0.666667
negative_log_loss   -0.484672
dtype: float64